In [1]:
from spark_utils import SparkUtils

# Create Spark session
su = SparkUtils("Example - Extract info from JSON", "spark://spark-master:7077")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 01:02:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
json_schema = SparkUtils.generate_schema([("id", "int"), ("json_col", "string")])
json_data = [
    (1, '{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}'),
    (2, '{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}'),
    (3, '{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}')
]

df_json = su._spark.createDataFrame(json_data, json_schema)
df_json.show(truncate=False)


+---+--------------------------------------------------------------------------------------------------------+
|id |json_col                                                                                                |
+---+--------------------------------------------------------------------------------------------------------+
|1  |{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}|
|2  |{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}                          |
|3  |{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}                          |
+---+--------------------------------------------------------------------------------------------------------+



In [3]:
from pyspark.sql.functions import get_json_object
df_json = df_json.withColumn("name", get_json_object(df_json.json_col, "$.name"))
df_json.show()

+---+--------------------+-------+
| id|            json_col|   name|
+---+--------------------+-------+
|  1|{"name": "Alice",...|  Alice|
|  2|{"name": "Bob", "...|    Bob|
|  3|{"name": "Charlie...|Charlie|
+---+--------------------+-------+



In [4]:
df_json = df_json.withColumn("age", get_json_object(df_json.json_col, "$.age"))
df_json.show()

+---+--------------------+-------+---+
| id|            json_col|   name|age|
+---+--------------------+-------+---+
|  1|{"name": "Alice",...|  Alice| 25|
|  2|{"name": "Bob", "...|    Bob| 30|
|  3|{"name": "Charlie...|Charlie| 35|
+---+--------------------+-------+---+



In [5]:
df_json = df_json.withColumn("city", get_json_object(df_json.json_col,"$.address.city"))


df_json.show()

+---+--------------------+-------+---+-----------+
| id|            json_col|   name|age|       city|
+---+--------------------+-------+---+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|
+---+--------------------+-------+---+-----------+



In [6]:
df_json = df_json.withColumn("1st_payment", get_json_object(df_json.json_col,"$.payment[0]"))

df_json.show()

[Stage 14:====================================>                   (11 + 6) / 17]

+---+--------------------+-------+---+-----------+-----------+
| id|            json_col|   name|age|       city|1st_payment|
+---+--------------------+-------+---+-----------+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|       NULL|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|       NULL|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|       NULL|
+---+--------------------+-------+---+-----------+-----------+



In [7]:
df_json = df_json.fillna({"1st_payment":0})
df_json.show()

[Stage 17:================================>                       (10 + 7) / 17]

+---+--------------------+-------+---+-----------+-----------+
| id|            json_col|   name|age|       city|1st_payment|
+---+--------------------+-------+---+-----------+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|          0|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|          0|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|          0|
+---+--------------------+-------+---+-----------+-----------+



In [8]:
from pyspark.sql.functions import from_json
# Deine the schema of the JSON object
people_schema = SparkUtils.generate_schema([("name", "string"),
                                            ("age", "int"),
                                            ("payments", "array_int"),
                                            ("address", "struct")])
df_parsed = df_json.withColumn("parsed", from_json(df_json.json_col, people_schema))
df_parsed.printSchema()
df_parsed.show(truncate=False)

root
 |-- id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- city: string (nullable = true)
 |-- 1st_payment: string (nullable = false)
 |-- parsed: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)
 |    |-- payments: array (nullable = true)
 |    |    |-- element: integer (containsNull = true)
 |    |-- address: struct (nullable = true)

+---+--------------------------------------------------------------------------------------------------------+-------+---+-----------+-----------+------------------------------+
|id |json_col                                                                                                |name   |age|city       |1st_payment|parsed                        |
+---+--------------------------------------------------------------------------------------------------------+-------+---+-----------+-----------+----